# 01 - Data exploration

**Goal.** Establish what the dataset actually contains, from the files
themselves rather than from any assumption, before a single line of modelling
code is written.

Questions answered here:

1. How many cycles, sensors and raw values are there?
2. What does one operating cycle look like?
3. How are the four component conditions distributed?
4. Are the cycles independent, or does their ordering matter?

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/; the modules live in src/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
print("project root:", PROJECT_ROOT)

## 1.1 Dataset structure, read off disk

In [ ]:
from data_loader import (
    SENSOR_SPECS, SENSOR_NAMES, TARGET_SPECS,
    get_dataset_summary, load_profile, load_sensor_matrix, print_dataset_summary,
)

summary = print_dataset_summary()

Every figure above is read from the files, then cross-checked against the
dimensions stated in the dataset's own `documentation.txt`. `load_sensor_matrix`
raises if a file's width contradicts the documented sample count, so a corrupted
or wrong-version download fails loudly instead of silently producing nonsense.

In [ ]:
# The dataset ships its own documentation. Read it rather than trusting a summary.
from data_loader import load_documentation_text
print(load_documentation_text()[:2500])

## 1.2 What does one cycle look like?

In [ ]:
# One cycle = 60 seconds of a repeated constant-load programme.
# Note the three different sampling rates: the x-axis is seconds, not samples.
fig, axes = plt.subplots(6, 1, figsize=(11, 12))
for ax, sensor in zip(axes, ["PS1", "PS3", "EPS1", "FS1", "TS1", "CE"]):
    matrix = load_sensor_matrix(sensor)
    values = matrix[0]
    ax.plot(np.linspace(0, 60, len(values)), values, linewidth=0.7)
    ax.set_ylabel(f"{sensor}")
    ax.margins(x=0)
    del matrix
axes[-1].set_xlabel("time within cycle [s]")
fig.suptitle("Cycle 0 across sensors of different sampling rates")
fig.tight_layout()

**This is the figure that motivates the whole project.** A cycle is a structured
programme with load steps, not a static reading. Collapsing it to a single mean
throws away the shape in which the fault signatures live — which is exactly what
the baseline experiment does, so that the engineered features have something
honest to be compared against.

## 1.3 Do the same sensors look different under different fault levels?

In [ ]:
profile = load_profile()
spec = TARGET_SPECS["cooler"]

# Overlay one cycle from each cooler condition level.
matrix = load_sensor_matrix("CE")
fig, ax = plt.subplots(figsize=(10, 4))
for value in sorted(spec.value_labels):
    idx = profile.index[profile[spec.column] == value][0]
    ax.plot(matrix[idx], label=f"{value} % - {spec.value_labels[value]}", linewidth=1.2)
ax.set_xlabel("sample within cycle (1 Hz)")
ax.set_ylabel("CE - cooling efficiency [%]")
ax.legend()
ax.set_title("Cooling efficiency separates the cooler condition levels almost completely")
del matrix

The cooler classes are visibly separated in a single channel. That predicts —
correctly, as the results later confirm — that the cooler is an *easy* target and
that feature engineering will add little to it. The dataset's own documentation
says the same: cooler and valve were "easy" targets in the original paper, with
pump and especially accumulator harder.

## 1.4 Class distributions and imbalance

In [ ]:
for key, spec in TARGET_SPECS.items():
    counts = profile[spec.column].value_counts().sort_index()
    ratio = counts.max() / counts.min()
    print(f"\n{spec.display_name} [{spec.unit}]  imbalance ratio = {ratio:.2f}")
    for value, count in counts.items():
        print(f"   {value:>5} = {spec.value_labels.get(int(value), ''):<30}"
              f"{count:>5} cycles ({100 * count / counts.sum():5.1f} %)")

The worst imbalance is roughly 3:1 (valve: 51 % of cycles in one class). That is
mild — mild enough that **class weighting is appropriate and resampling is not**.
It is not mild enough to trust plain accuracy, which is why macro F1 is the
headline metric everywhere in this project.

## 1.5 Are the cycles independent? (the split question)

In [ ]:
# The cycles are stored in the order they were recorded. Plot the condition
# level against cycle number to see whether the experiment was randomised.
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
for ax, (key, spec) in zip(axes, TARGET_SPECS.items()):
    ax.plot(profile["cycle_id"], profile[spec.column], linewidth=0.8)
    ax.set_ylabel(spec.display_name.split()[0])
axes[-1].set_xlabel("cycle number (chronological)")
fig.suptitle("Component conditions were varied as a designed sequence, not randomly")
fig.tight_layout()

In [ ]:
# Consequence: check what a chronological 80/20 split would leave in the test set.
cut = int(0.8 * len(profile))
for key, spec in TARGET_SPECS.items():
    train_classes = sorted(profile[spec.column][:cut].unique())
    test_classes = sorted(profile[spec.column][cut:].unique())
    print(f"{spec.display_name:<26} train: {train_classes}   test: {test_classes}")

**Finding that shapes the whole experimental design.** The final 20 % of cycles
contain only **one** cooler class. A chronological holdout would therefore leave
entire classes untestable, so it is not a usable protocol on this dataset.

Stratified random splitting is used instead. But because neighbouring cycles were
recorded moments apart under the same component setting, a random split can place
near-duplicate cycles on both sides. That residual optimism is measured
separately in notebook 04 with a block-grouped cross-validation.